In [ ]:
#mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#install libraries
!pip install -q torch transformers pillow matplotlib requests

In [ ]:
import requests
from PIL import Image
import torch
import matplotlib.pyplot as plt
from transformers import DepthProImageProcessorFast, DepthProForDepthEstimation

image_path = '/image_path.jpg'  #give image path
image = Image.open(image_path)

# Load huggingface model and processor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_processor = DepthProImageProcessorFast.from_pretrained("apple/DepthPro-hf")
model = DepthProForDepthEstimation.from_pretrained("apple/DepthPro-hf").to(device)

# Preprocess image
inputs = image_processor(images=image, return_tensors="pt").to(device)

# Inference
with torch.no_grad():
    outputs = model(**inputs)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.90G [00:00<?, ?B/s]

In [ ]:
import pandas as pd
post_processed_output = image_processor.post_process_depth_estimation(
    outputs, target_sizes=[(image.height, image.width)]
)
depth = post_processed_output[0]["predicted_depth"]  # Fixed line

output_csv_path = '/output_path.csv'  # change this if needed

# Save CSV: Convert to NumPy array
depth_np = depth.squeeze().cpu().numpy()  # shape: (H, W)
pd.DataFrame(depth_np).to_csv(output_csv_path, index=False, header=False)

print(f"Depth map saved to: {output_csv_path}")

# Visualize metric depth map (normalized for display)
depth_vis = (depth - depth.min()) / (depth.max() - depth.min())
depth_vis = depth.cpu().numpy().squeeze()

plt.figure(figsize=(10, 10))
plt.imshow(depth_vis, cmap='viridis')
plt.colorbar(label='Depth (metres)')
plt.title('Metric Depth Visualization')
plt.axis('off')
plt.show()

# To access the raw metric depth (in meters), use `depth` before normalization
print("Raw metric depth min/max (meters):", depth.min().item(), depth.max().item())

In [ ]:
import numpy as np
np.save("depthsaved", depth_np) #save as .npy file

In [ ]:
# --- Save plotted visualization with colorbar and title ---
fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(depth_vis, cmap='viridis')
cbar = plt.colorbar(im, ax=ax, label='Depth (metres)')
ax.set_title('Metric Depth Visualization')
ax.axis('off')

# Save full plot with scale
plot_output_path = f'/path/plot.png' #specify path
plt.savefig(plot_output_path, bbox_inches='tight', pad_inches=0.1)
plt.close(fig)  # Close to avoid duplicate display in notebook

print(f"Full plot with colorbar saved to: {plot_output_path}")

For UnidepthV2 implementation follow the link:
https://www.kaggle.com/code/bhavishaa/monocular-depth-estimation-guide-for-beginners